In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

In [ ]:
# ===============================
# CONFIG
# ===============================

MODEL_NAME = "microsoft/deberta-v3-base"
NUM_LABELS = 6
BATCH_SIZE = 8
EPOCHS = 3
KFOLDS = 5
LR = 2e-5
MAX_LEN = 512

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_dir = "/content/drive/MyDrive/multilabel-classification/working/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)


In [ ]:
# ===============================
# LOAD DATA
# ===============================

df = pd.read_csv('/content/drive/MyDrive/multilabel-classification/dataset/train.csv')

label_columns = [
    'Computer Science',
    'Physics',
    'Mathematics',
    'Statistics',
    'Quantitative Biology',
    'Quantitative Finance'
]

df[label_columns] = df[label_columns].fillna(0).astype(float)

titles = df['TITLE'].fillna('').astype(str).tolist()
abstracts = df['ABSTRACT'].fillna('').astype(str).tolist()
labels = df[label_columns].values

In [ ]:
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# ===============================
# TOKENIZER
# ===============================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# ===============================
# DATASET CLASS
# ===============================

class MultiLabelDataset(torch.utils.data.Dataset):

    def __init__(self, titles, abstracts, labels, tokenizer, max_len):
        self.titles = titles
        self.abstracts = abstracts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):

        text = self.titles[idx] + " " + self.abstracts[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)

        return item


In [ ]:
# ===============================
# LOSS FUNCTION
# ===============================

criterion = nn.BCEWithLogitsLoss()


In [ ]:
# =========================================================
# SETTINGS
# =========================================================

import os
import sys
checkpoint_dir = "/content/drive/MyDrive/multilabel-classification/working/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

results_path = "/content/drive/MyDrive/multilabel-classification/working/kfold_results.csv"

torch.backends.cudnn.benchmark = True


# =========================================================
# TRAIN FUNCTION
# =========================================================

def train_epoch(model, dataloader, optimizer):

    model.train()
    total_loss = 0

    all_preds = []
    all_labels = []

    for step, batch in enumerate(dataloader):

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE).float()

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

        # =================================================
        # SAME LINE PROGRESS UPDATE
        # =================================================

        progress = int((step + 1) / len(dataloader) * 100)

        sys.stdout.write(
            f"\rFold {CURRENT_FOLD} | Epoch {CURRENT_EPOCH}/{EPOCHS} | Progress {progress}% | Loss {loss.item():.4f}"
        )
        sys.stdout.flush()

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int()

        all_preds.append(preds.detach().cpu())
        all_labels.append(labels.detach().cpu().int())

    print()  # move cursor to next line after epoch finishes

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    micro_f1 = f1_score(all_labels, all_preds, average='micro')
    macro_f1 = f1_score(all_labels, all_preds, average='macro')

    precision = precision_score(all_labels, all_preds, average='micro')
    recall = recall_score(all_labels, all_preds, average='micro')

    subset_acc = accuracy_score(all_labels, all_preds)

    label_acc = accuracy_score(
        all_labels.flatten(),
        all_preds.flatten()
    )

    avg_loss = total_loss / len(dataloader)

    return avg_loss, subset_acc, label_acc, precision, recall, micro_f1, macro_f1


# =========================================================
# VALIDATION FUNCTION
# =========================================================

def eval_model(model, dataloader):

    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE).float()

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            loss = criterion(logits, labels)

            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu().int())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    micro_f1 = f1_score(all_labels, all_preds, average='micro')
    macro_f1 = f1_score(all_labels, all_preds, average='macro')

    precision = precision_score(all_labels, all_preds, average='micro')
    recall = recall_score(all_labels, all_preds, average='micro')

    subset_acc = accuracy_score(all_labels, all_preds)

    label_acc = accuracy_score(
        all_labels.flatten(),
        all_preds.flatten()
    )

    avg_loss = total_loss / len(dataloader)

    return avg_loss, subset_acc, label_acc, precision, recall, micro_f1, macro_f1


# =========================================================
# STORE FOLD RESULTS
# =========================================================

fold_results = {
    "Subset Accuracy": [],
    "Label Accuracy": [],
    "Precision": [],
    "Recall": [],
    "F1 Micro": [],
    "F1 Macro": [],
    "Val Loss": []
}

kf = KFold(n_splits=KFOLDS, shuffle=True, random_state=42)


# =========================================================
# TRAINING LOOP
# =========================================================

for fold, (train_idx, val_idx) in enumerate(kf.split(titles)):

    checkpoint_path = f"{checkpoint_dir}/deberta_fold{fold+1}_last.pt"

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        if checkpoint["epoch"] >= EPOCHS - 1:
            print(f"Fold {fold+1} already completed. Skipping.")
            continue

    print(f"\n========== FOLD {fold+1}/{KFOLDS} ==========")

    train_titles = [titles[i] for i in train_idx]
    train_abstracts = [abstracts[i] for i in train_idx]
    train_labels = labels[train_idx]

    val_titles = [titles[i] for i in val_idx]
    val_abstracts = [abstracts[i] for i in val_idx]
    val_labels = labels[val_idx]

    train_dataset = MultiLabelDataset(train_titles, train_abstracts, train_labels, tokenizer, MAX_LEN)
    val_dataset = MultiLabelDataset(val_titles, val_abstracts, val_labels, tokenizer, MAX_LEN)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=True,
        num_workers=2
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        pin_memory=True,
        num_workers=2
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification"
    )

    model = model.float()
    model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    start_epoch = 0
    best_f1 = 0

    if os.path.exists(checkpoint_path):

        print("Checkpoint found. Loading...")

        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        start_epoch = checkpoint["epoch"] + 1
        best_f1 = checkpoint.get("best_f1", 0)

        print(f"Resuming Fold {fold+1} from Epoch {start_epoch}")

    # =====================================================
    # EPOCH LOOP
    # =====================================================

    for epoch in range(start_epoch, EPOCHS):

        CURRENT_FOLD = fold + 1
        CURRENT_EPOCH = epoch + 1

        print(f"\nEpoch {epoch+1}/{EPOCHS} Fold {fold+1}")
        print("="*50)

        train_loss, subset_acc, label_acc, precision, recall, f1_micro, f1_macro = train_epoch(
            model, train_loader, optimizer
        )

        print("\nTRAIN METRICS")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Subset Accuracy: {subset_acc:.4f}")
        print(f"Label Accuracy: {label_acc:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1 Micro: {f1_micro:.4f}")
        print(f"F1 Macro: {f1_macro:.4f}")

        val_loss, subset_acc, label_acc, precision, recall, f1_micro, f1_macro = eval_model(
            model, val_loader
        )

        print("\nVALIDATION METRICS")
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Subset Accuracy: {subset_acc:.4f}")
        print(f"Label Accuracy: {label_acc:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1 Micro: {f1_micro:.4f}")
        print(f"F1 Macro: {f1_macro:.4f}")

        if f1_micro > best_f1:

            best_f1 = f1_micro

            best_path = f"{checkpoint_dir}/best_fold{fold+1}.pt"

            torch.save({
                "model_state_dict": model.state_dict(),
                "f1_micro": best_f1,
                "fold": fold
            }, best_path)

            print(f"Best model saved → {best_path}")

        torch.save({
            "fold": fold,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_f1": best_f1
        }, checkpoint_path)

        print(f"Checkpoint saved → {checkpoint_path}")

        if epoch == EPOCHS - 1:

            fold_results["Subset Accuracy"].append(subset_acc)
            fold_results["Label Accuracy"].append(label_acc)
            fold_results["Precision"].append(precision)
            fold_results["Recall"].append(recall)
            fold_results["F1 Micro"].append(f1_micro)
            fold_results["F1 Macro"].append(f1_macro)
            fold_results["Val Loss"].append(val_loss)

    results_df = pd.DataFrame(fold_results)
    results_df.to_csv(results_path, index=False)

    print(f"Results saved → {results_path}")

In [ ]:
# =========================================================
# FINAL RESULTS
# =========================================================

print("\n========== FINAL K-FOLD RESULTS ==========")

results_df = pd.DataFrame(fold_results)

print(results_df)

print("\nAverage Results")

print(results_df.mean())

In [ ]:
# ===============================
# PRINT FOLD RESULTS
# ===============================

print("\n=============== FOLD RESULTS ================\n")

results_df = pd.DataFrame(fold_results)
results_df.index = [f"Fold {i+1}" for i in range(len(results_df))]
print(results_df)

# ===============================
# CROSS VALIDATION SUMMARY
# ===============================

print("\n================ CROSS-VALIDATION SUMMARY ================\n")

cv_summary = {}

for col in results_df.columns:

    mean = results_df[col].mean()
    std = results_df[col].std()

    cv_summary[col] = f"{mean:.4f} ± {std:.4f}"

summary_df = pd.DataFrame.from_dict(
    cv_summary,
    orient="index",
    columns=["CV Result (Mean ± Std)"]
)

print(summary_df)